## Overview

In this notebook, we will utilize the AIEnrichment Service to execute the MedImageParse model, transforming it into the standard AIEnrichment Output schema. This process includes several steps: configuration management, model setup, input preparation, and execution of the enrichment service.

### Prerequisites
- **MedImageParse healthcare AI Model Deployment:** Ensure you have a deployed MedImageParse model. Refer to the MedImageParse healthcare AI model documentation [here](https://learn.microsoft.com/en-us/azure/ai-studio/how-to/healthcare-ai/deploy-medimageparse).


- **DICOM data transformation capability:** Ensure you have the DICOM data transformation capability set up. Run the pipeline to ingest & transform DICOM (imaging) data into the silver lakehouse. Refer to the DICOM data transformation capability documentation [here](https://learn.microsoft.com/en-us/industry/healthcare/healthcare-data-solutions/dicom-data-transformation-overview).

### Input Preparation
- Define the Enrichment Definition to prepare the enrichment input data.
- Specify metadata and image file references for the enrichment process.

### AI Enrichment Service Execution
- Execute the enrichment service using the Enrichment ID.

### Configurations
Before running this notebook, ensure that all configurations in the msft_config_notebook are completed.

###### Configuration management and setup

To setup and manage configurations for the Healthcare data solutions, please execute the following cell:

In [ ]:
%run msft_config_notebook

In [ ]:
%run msft_config_notebook {"enable_spark_setup" : true, "enable_packages_mount" : false}

In [ ]:
from microsoft.fabric.hls.hds.ai_enrichments.use_cases import MedImageParseTransformer,MedImageParseProcessor
from microsoft.fabric.hls.hds.ai_enrichments.core.services.ai_enrichments_service import AIEnrichmentsService
from microsoft.fabric.hls.hds.ai_enrichments.core import EnrichmentView,Enrichment,EnrichmentViewExpression,EnrichmentDefinition,EnrichmentInputMapping,EnrichmentFileReference,EnrichmentViewDefinition

##### Configuration for MedImageParse Enrichment

In [ ]:
# Inline Parameters
INLINES_PARAMS={ 
        'refresh-metadata':True # Set to TRUE on the initial run or whenever new tables are added to silver
}

#### Initialize Processor and Transformer

In [ ]:
med_image_parse_processor=MedImageParseProcessor(execution_batch_size=1,max_workers=1)
med_image_parse_transformer=MedImageParseTransformer()

#### Initialize AIEnrichment Service

In [ ]:
ai_enrichments_service=AIEnrichmentsService(
        spark,
        workspace_name=workspace_name,
        solution_name=solution_name,
        admin_lakehouse_name=administration_database_name,
        inline_params=INLINES_PARAMS,
        enrichment_model_processor=med_image_parse_processor,
        enrichment_transformer=med_image_parse_transformer,
        one_lake_endpoint=one_lake_endpoint)

### AIEnrichment Definition

##### Create View Definition
Refer to [DICOM metadata transformation mapping](https://learn.microsoft.com/en-us/industry/healthcare/healthcare-data-solutions/dicom-data-transformation-mapping#transformation-mapping-for-bronze-to-silver-delta-table) to learn more about the DICOM metadata tags.

In [ ]:
# METADATA DEFINITION VARIABLES
ENRICHMENT_VIEW_TABLES = ["ImagingMetastore"]

# Query to get patient name, patient Id from the metadata column and the formatted filename to be used as document Id
# DICOM tags used in the query: 
# 00100010 - patient name           00100020 - patient Id               00100040 - patient sex                   
# 00080020 - study date             00180015 - body part examined     

SQL_QUERY = """
SELECT 
    COALESCE(
        get_json_object(metadata['00100010'].Value[0], '$.Alphabetic'), 
        metadata['00100010'].Value[0]
    ) AS patient_name, 
    metadata['00100020'].Value[0] AS patient_id,
    regexp_extract(IM.filePath, '([^/]+)$', 1) AS document_id, 
    IM.filePath AS content 
FROM 
    view1 AS IM 
WHERE 
    to_date(metadata['00080020'].value[0], 'yyyyMMdd') > to_date('20000223', 'yyyyMMdd') and
    metadata['00180015'].value[0] == 'LUNG' and
    metadata['00100040'].value[0] == 'M' and
    metadata['00100020'].Value[0] IS NOT NULL 
"""

PARENT_VIEW_IDS = ai_enrichments_service.metadata.get_enrichment_view_ids(ENRICHMENT_VIEW_TABLES)

# Define the SQL expression for the enrichment view
sql_expression = EnrichmentViewExpression(
    type="sql",  # Type of the expression (e.g., SQL)
    query=SQL_QUERY
)

# Define the enrichment view definition
enrichment_view_definition = EnrichmentViewDefinition(
    parent_views_ids=PARENT_VIEW_IDS,  # List of parent view IDs (if any)
    expression=sql_expression  # SQL expression for the enrichment view
)

  # Create an instance of EnrichmentView  
enrichment_view = EnrichmentView(      
    name='View for MedImageParse Data',  
    description='View for extracting data for MedImageParse enrichment',  
    definition=enrichment_view_definition
)  

enrichment_view_id=ai_enrichments_service.metadata.create_enrichment_view(enrichment_view)

##### Create Enrichment Definition

In [ ]:
#MedImageParse API Configuration
MED_IMAGE_PARSE_API_KEY_SECRET_NAME = ""  
MED_IMAGE_PARSE_API_ENDPOINT = ""  
MED_IMAGE_PARSE_MODEL_VERSION = ""  
MED_IMAGE_PARSE_SYSTEM_INSTRUCTIONS=""
MED_IMAGE_PARSE_MODEL_NAME="MedImageParse"

model_definition = {  
    "api_key_secret_name": MED_IMAGE_PARSE_API_KEY_SECRET_NAME,  
    "api_endpoint": MED_IMAGE_PARSE_API_ENDPOINT,  
    "version": MED_IMAGE_PARSE_MODEL_VERSION,
    "name":MED_IMAGE_PARSE_MODEL_NAME,
    "system_instructions":MED_IMAGE_PARSE_SYSTEM_INSTRUCTIONS
}  

In [ ]:
# Define the metadata column definition
meta_data_column_definition={ 
    "document_id":"document_id", # Add metadata column definitions here
    "patient_name":"patient_name"
}


# Define the column references for the enrichment input  
column_references=EnrichmentFileReference(  
                    id="document_id",   # ID of the file reference
                    content="content",  # Content of the file reference
                )

# Define the input mapping for the enrichment
input_mapping = EnrichmentInputMapping(  
    patient_id="patient_id",                        # Mapping for patient ID
    metadata=meta_data_column_definition,           # Metadata column definitions
    image_resource_references=[column_references]   # List of image resource references
)  

# Create an instance of Enrichment
enrichment = Enrichment(  
    name='MedImageParse Enrichment',  # Name of the enrichment
    description='Definition for MedImageParse Enrichment',  # Description of the enrichment
    definition=EnrichmentDefinition(  
        model=model_definition,         # Model definition
        view_id=f"{enrichment_view_id}",# View ID for the enrichment
        input_mapping=input_mapping     # Input mapping for the enrichment
    )  
)  

enrichment_id=ai_enrichments_service.metadata.create_enrichment(enrichment)

### AIEnrichment Execution

In [ ]:
ai_enrichments_service.execution.execute(enrichment_id)